# Notebook 5: zero-shot generative spam classification

This notebook evaluates whether an instruction-tuned generative model can classify the fixed SMS and Enron test sets without project-specific training. It adds a different kind of evidence to the supervised TF-IDF, TextCNN, and DistilBERT experiments; it does not replace them.

## Predeclared protocol

- One public checkpoint: google/flan-t5-base.
- One fixed prompt and the verbalizers ham and spam; there is no prompt search.
- No train or validation examples are shown to the model.
- The primary prediction uses mean decoder-token log probability, including EOS, so the differently tokenized ham and spam labels remain comparable.
- Spam-class F1 receives a 95% stratified bootstrap interval. This measures test-sample uncertainty, not training-seed variability, because there is no training run.
- Greedy free generation is audited on 25 ham and 25 spam examples per domain. Invalid answers remain in the denominator and count as errors.

The model is a domain-agnostic zero-shot comparator. Therefore, a difference between SMS and Enron is called a domain performance gap, not a transfer gap.

In [ ]:
from pathlib import Path
import gc
import hashlib
import subprocess
import sys
import time

REPOSITORY_URL = 'https://github.com/kbozukov-coke/cross-domain-spam-detection.git'
PROJECT_NAME = 'cross-domain-spam-detection'
IS_KAGGLE = Path('/kaggle/working').exists()

if IS_KAGGLE:
    PROJECT_ROOT = Path('/kaggle/working') / PROJECT_NAME
    if not (PROJECT_ROOT / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)],
            check=True,
        )
    else:
        subprocess.run(
            ['git', '-C', str(PROJECT_ROOT), 'pull', '--ff-only'],
            check=True,
        )
    subprocess.run(
        [
            sys.executable, '-m', 'pip', 'install', '-q',
            'transformers==4.57.6', 'accelerate>=1.10,<2',
            'sentencepiece>=0.2,<1',
        ],
        check=True,
    )
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import transformers
from sklearn.metrics import ConfusionMatrixDisplay
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

from src.controls import subsample_to_class_counts
from src.data import load_prepared_splits, summarize_splits
from src.evaluation import (
    binary_classification_metrics,
    build_prediction_table,
    calibration_table,
    stratified_bootstrap_ci,
)
from src.generative import (
    confidence_coverage_table,
    format_prompts,
    generate_label_responses,
    generation_audit_summary,
    score_candidate_labels,
)
from src.modeling import run_transfer_experiments
from src.protocol import (
    BOOTSTRAP_SEED,
    CONTROL_SAMPLING_SEED,
    DATA_SPLIT_SEED,
    REFERENCE_TRAINING_SEED,
)

sns.set_theme(style='whitegrid', context='notebook')
pd.set_option('display.max_colwidth', 120)

print(f'Running in: {"Kaggle" if IS_KAGGLE else "local environment"}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(
    'GPU devices: '
    f'{[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]}'
)

if not torch.cuda.is_available():
    raise RuntimeError(
        'This notebook requires a GPU. Enable a Kaggle GPU accelerator and rerun.'
    )

## Load the fixed test data

The same cleaning and split seed are reused. Train and validation labels are loaded only because the shared data helper returns all splits; the zero-shot model receives only the two test sets.

In [ ]:
splits, cleaning_audit = load_prepared_splits(random_state=DATA_SPLIT_SEED)
split_summary = summarize_splits(splits)
split_summary.loc[
    split_summary['split'].eq('test'),
    ['dataset', 'split', 'rows', 'ham', 'spam', 'spam_rate'],
]

## Locked model, prompt, and inference settings

The complete instruction appears before the message. Right truncation can therefore remove only the tail of a long message, not the class definitions or requested output format. The prompt ID and SHA-256 hash make later changes visible.

In [ ]:
MODEL_NAME = 'google/flan-t5-base'
MODEL_REVISION = '7bcac572ce56db69c1ea7c8af255c5d7c9672fc2'
MODEL_LABEL = 'FLAN-T5 Base (zero-shot)'
PROMPT_ID = 'spam_definition_v1'
PROMPT_TEMPLATE = """You are an SMS and email spam classifier.
spam means unsolicited advertising, fraud, phishing, or a deceptive request.
ham means a legitimate non-spam message.
Return exactly one word: ham or spam.

Message:
{text}"""
PROMPT_SHA256 = hashlib.sha256(PROMPT_TEMPLATE.encode('utf-8')).hexdigest()

MAX_LENGTH = 512
BATCH_SIZE = 8
MAX_NEW_TOKENS = 4
AUDIT_ROWS_PER_CLASS = 25
N_BOOTSTRAP = 2_000
SOURCE_NOTEBOOK = 'notebooks/05_zero_shot_generative.ipynb'

configuration = pd.Series(
    {
        'model_name': MODEL_NAME,
        'model_revision': MODEL_REVISION,
        'prompt_id': PROMPT_ID,
        'prompt_sha256': PROMPT_SHA256,
        'max_length': MAX_LENGTH,
        'batch_size': BATCH_SIZE,
        'max_new_tokens': MAX_NEW_TOKENS,
        'audit_rows_per_class_and_domain': AUDIT_ROWS_PER_CLASS,
        'bootstrap_repetitions': N_BOOTSTRAP,
        'bootstrap_seed': BOOTSTRAP_SEED,
        'audit_sampling_seed': CONTROL_SAMPLING_SEED,
    },
    name='value',
)
configuration.to_frame()

## Load FLAN-T5

Only cuda:0 is used. FP16 and sequential batched inference keep memory bounded on a single T4. The model is placed in evaluation mode and is never updated.

In [ ]:
device = torch.device('cuda:0')
torch.cuda.set_device(device)
torch.cuda.reset_peak_memory_stats(device)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
)
tokenizer.truncation_side = 'right'
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True,
    use_safetensors=True,
).to(device)
model.eval()

parameter_count = sum(parameter.numel() for parameter in model.parameters())
prompt_prefix_tokens = len(
    tokenizer(
        PROMPT_TEMPLATE.format(text=''),
        add_special_tokens=True,
    )['input_ids']
)
if prompt_prefix_tokens >= MAX_LENGTH:
    raise RuntimeError('The fixed instruction does not fit inside MAX_LENGTH.')

print(f'Device used: {device} ({torch.cuda.get_device_name(device)})')
print(f'Parameters: {parameter_count:,}')
print(f'Fixed prompt tokens including EOS: {prompt_prefix_tokens}')

## Two-example preflight

Before the full pass, this cell checks the Kaggle-only model API, records the verbalizer token lengths, and exercises both teacher-forced scoring and greedy generation. It does not choose the prompt or estimate performance.

In [ ]:
candidate_token_lengths = {
    label: len(
        tokenizer(
            text_target=label,
            add_special_tokens=True,
        )['input_ids']
    )
    for label in ('ham', 'spam')
}
if any(length <= 0 for length in candidate_token_lengths.values()):
    raise RuntimeError(
        f'Invalid verbalizer token lengths: {candidate_token_lengths}'
    )
if len(set(candidate_token_lengths.values())) != 1:
    print(
        'Verbalizers have different token lengths; candidate scores use '
        'mean token log probability including EOS.'
    )

preflight_texts = [
    'Hello, are we still meeting at six?',
    'Claim your free cash prize now.',
]
preflight_scores = score_candidate_labels(
    model,
    tokenizer,
    preflight_texts,
    prompt_template=PROMPT_TEMPLATE,
    max_length=MAX_LENGTH,
    batch_size=2,
    device=device,
)
preflight_generation = generate_label_responses(
    model,
    tokenizer,
    preflight_texts,
    prompt_template=PROMPT_TEMPLATE,
    max_length=MAX_LENGTH,
    batch_size=2,
    max_new_tokens=MAX_NEW_TOKENS,
    device=device,
)
if not np.isfinite(
    preflight_scores[['ham_score', 'spam_score', 'spam_probability']]
).to_numpy().all():
    raise RuntimeError('The preflight produced a non-finite score.')

print(f'Verbalizer token lengths: {candidate_token_lengths}')
pd.concat(
    [
        pd.Series(preflight_texts, name='message'),
        preflight_scores[['spam_probability', 'prediction']],
        preflight_generation[['generated_text', 'generated_label', 'valid_generation']],
    ],
    axis=1,
)

## Length-normalized restricted-label scores on the full test sets

For each message, the model scores the complete ham and spam decoder sequences, including EOS. Because the tokenizer produces three target tokens for ham and two for spam, each sequence score is divided by its non-padding token count. Softmax over the two mean scores produces a score-derived spam pseudo-probability. It is useful for ranking and confidence analysis, but it is neither a literal sequence probability nor globally normalized over every possible generation.

In [ ]:
test_frames = {
    domain: splits[domain]['test'].reset_index(drop=True)
    for domain in ('sms', 'enron')
}
prediction_tables = {}
metric_rows = []
domain_runtime_seconds = {}

for domain, test_frame in test_frames.items():
    prompts = format_prompts(test_frame['text'].tolist(), PROMPT_TEMPLATE)
    input_token_lengths = []
    for start in range(0, len(prompts), 128):
        tokenized_lengths = tokenizer(
            prompts[start:start + 128],
            add_special_tokens=True,
            truncation=False,
            padding=False,
            return_length=True,
            verbose=False,
        )['length']
        input_token_lengths.extend(int(length) for length in tokenized_lengths)
    input_token_lengths = np.asarray(input_token_lengths, dtype=np.int32)

    torch.cuda.synchronize(device)
    started = time.perf_counter()
    label_scores = score_candidate_labels(
        model,
        tokenizer,
        test_frame['text'].tolist(),
        prompt_template=PROMPT_TEMPLATE,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE,
        device=device,
        verbose=True,
    )
    torch.cuda.synchronize(device)
    elapsed = time.perf_counter() - started
    domain_runtime_seconds[domain] = elapsed

    details = build_prediction_table(
        test_frame,
        label_scores['spam_probability'],
        model=MODEL_LABEL,
        training_seed=None,
        train_domain='none',
        test_domain=domain,
        setting='zero-shot',
    )
    details['model_name'] = MODEL_NAME
    details['model_revision'] = MODEL_REVISION
    details['prompt_id'] = PROMPT_ID
    details['prompt_sha256'] = PROMPT_SHA256
    details['ham_score'] = label_scores['ham_score'].to_numpy()
    details['spam_score'] = label_scores['spam_score'].to_numpy()
    details['label_score_margin'] = label_scores['label_score_margin'].to_numpy()
    details['label_score_normalization'] = (
        'mean_token_log_probability_including_eos'
    )
    details['input_tokens'] = input_token_lengths
    details['truncated'] = input_token_lengths > MAX_LENGTH
    details['source_notebook'] = SOURCE_NOTEBOOK
    prediction_tables[domain] = details

    metrics = binary_classification_metrics(
        details['label'],
        details['spam_probability'],
    )
    interval = stratified_bootstrap_ci(
        details['label'],
        details['spam_probability'],
        metric='f1',
        n_bootstrap=N_BOOTSTRAP,
        random_state=BOOTSTRAP_SEED,
    )
    metric_rows.append(
        {
            'model': MODEL_LABEL,
            'model_name': MODEL_NAME,
            'model_revision': MODEL_REVISION,
            'prompt_id': PROMPT_ID,
            'prompt_sha256': PROMPT_SHA256,
            'training_seed': pd.NA,
            'train_domain': 'none',
            'test_domain': domain,
            'setting': 'zero-shot',
            'test_rows': len(details),
            **metrics,
            'f1_ci_low': interval['ci_low'],
            'f1_ci_high': interval['ci_high'],
            'f1_bootstrap_standard_error': interval['bootstrap_standard_error'],
            'bootstrap_repetitions': interval['n_bootstrap'],
            'bootstrap_seed': interval['random_state'],
            'ham_target_tokens': candidate_token_lengths['ham'],
            'spam_target_tokens': candidate_token_lengths['spam'],
            'label_score_normalization': (
                'mean_token_log_probability_including_eos'
            ),
            'mean_input_tokens': float(input_token_lengths.mean()),
            'max_input_tokens': int(input_token_lengths.max()),
            'truncated_rows': int((input_token_lengths > MAX_LENGTH).sum()),
            'truncation_rate': float((input_token_lengths > MAX_LENGTH).mean()),
            'inference_seconds': elapsed,
            'rows_per_second': len(details) / elapsed,
            'max_length': MAX_LENGTH,
            'batch_size': BATCH_SIZE,
            'parameters': parameter_count,
            'device': str(device),
            'gpu_name': torch.cuda.get_device_name(device),
            'source_notebook': SOURCE_NOTEBOOK,
        }
    )
    print(f'Finished {domain}: {len(details)} rows in {elapsed:.1f} seconds.')

zero_shot_results = pd.DataFrame(metric_rows)

In [ ]:
display_columns = [
    'test_domain', 'test_rows', 'accuracy', 'precision', 'recall', 'f1',
    'f1_ci_low', 'f1_ci_high', 'macro_f1', 'balanced_accuracy', 'mcc',
    'roc_auc', 'pr_auc', 'brier_score', 'ece', 'truncated_rows',
    'truncation_rate', 'inference_seconds',
]
zero_shot_results.loc[:, display_columns].round(3)

## Confidence and calibration

Reliability diagrams compare the score-derived spam pseudo-probability with the observed spam rate. The coverage table also shows what happens when low-confidence predictions are withheld. These are diagnostics, not a claim that the scores are calibrated probabilities.

In [ ]:
calibration_frames = []
coverage_frames = []

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for axis, domain in zip(axes, ('sms', 'enron')):
    details = prediction_tables[domain]
    calibration = calibration_table(
        details['label'],
        details['spam_probability'],
        n_bins=10,
    ).assign(
        model=MODEL_LABEL,
        test_domain=domain,
        setting='zero-shot',
        prompt_id=PROMPT_ID,
        source_notebook=SOURCE_NOTEBOOK,
    )
    calibration_frames.append(calibration)

    coverage = confidence_coverage_table(
        details['label'],
        details['spam_probability'],
    ).assign(
        model=MODEL_LABEL,
        test_domain=domain,
        setting='zero-shot',
    )
    coverage_frames.append(coverage)

    axis.plot([0, 1], [0, 1], linestyle='--', color='black', label='ideal')
    axis.plot(
        calibration['mean_probability'],
        calibration['observed_spam_rate'],
        marker='o',
        label=domain.upper(),
    )
    axis.set(
        title=f'{domain.upper()} reliability',
        xlabel='Mean score-derived spam pseudo-probability',
        ylabel='Observed spam rate',
        xlim=(0, 1),
        ylim=(0, 1),
    )
    axis.legend()

plt.tight_layout()
plt.show()

calibration_results = pd.concat(calibration_frames, ignore_index=True)
confidence_coverage = pd.concat(coverage_frames, ignore_index=True)
confidence_coverage.loc[
    :,
    ['test_domain', 'minimum_confidence', 'retained_rows', 'coverage', 'accuracy'],
].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for axis, domain in zip(axes, ('sms', 'enron')):
    details = prediction_tables[domain]
    ConfusionMatrixDisplay.from_predictions(
        details['label'],
        details['prediction'],
        display_labels=['ham', 'spam'],
        cmap='Blues',
        colorbar=False,
        ax=axis,
    )
    axis.set_title(f'{domain.upper()} zero-shot')
plt.tight_layout()
plt.show()

## Free-generation audit

The main experiment constrains the choice to two labels. This smaller audit checks whether greedy free generation actually follows the one-word instruction. The deterministic sample contains the same class counts in each domain. Invalid outputs are retained and counted as incorrect.

In [ ]:
audit_frames = []
audit_summary_rows = []

for domain, test_frame in test_frames.items():
    audit_frame = subsample_to_class_counts(
        test_frame,
        {0: AUDIT_ROWS_PER_CLASS, 1: AUDIT_ROWS_PER_CLASS},
        random_state=CONTROL_SAMPLING_SEED,
    )
    audit_identity = build_prediction_table(
        audit_frame,
        np.full(len(audit_frame), 0.5),
        model=MODEL_LABEL,
        training_seed=None,
        train_domain='none',
        test_domain=domain,
        setting='zero-shot',
    )
    likelihood_rows = (
        prediction_tables[domain]
        .set_index('example_id')
        .loc[audit_identity['example_id']]
        .reset_index()
    )
    if not np.array_equal(
        likelihood_rows['label'].to_numpy(),
        audit_identity['label'].to_numpy(),
    ):
        raise RuntimeError('Audit rows are not aligned with full-test predictions.')

    torch.cuda.synchronize(device)
    started = time.perf_counter()
    generated = generate_label_responses(
        model,
        tokenizer,
        audit_frame['text'].tolist(),
        prompt_template=PROMPT_TEMPLATE,
        max_length=MAX_LENGTH,
        batch_size=BATCH_SIZE,
        max_new_tokens=MAX_NEW_TOKENS,
        device=device,
        verbose=True,
    )
    torch.cuda.synchronize(device)
    audit_elapsed = time.perf_counter() - started

    audit = likelihood_rows.loc[
        :,
        [
            'example_id', 'test_domain', 'label', 'prediction',
            'spam_probability', 'source',
        ],
    ].rename(columns={'prediction': 'likelihood_prediction'})
    audit['generated_text'] = generated['generated_text'].to_numpy()
    audit['generated_label'] = generated['generated_label'].to_numpy()
    audit['generated_label_id'] = generated['generated_label_id'].to_numpy()
    audit['valid_generation'] = generated['valid_generation'].to_numpy()
    audit['generation_correct'] = audit['generated_label_id'].eq(audit['label'])
    audit['likelihood_generation_agreement'] = (
        audit['valid_generation']
        & audit['generated_label_id'].eq(audit['likelihood_prediction'])
    )
    audit['model'] = MODEL_LABEL
    audit['model_name'] = MODEL_NAME
    audit['model_revision'] = MODEL_REVISION
    audit['prompt_id'] = PROMPT_ID
    audit['source_notebook'] = SOURCE_NOTEBOOK
    audit_frames.append(audit)

    audit_summary_rows.append(
        {
            'test_domain': domain,
            **generation_audit_summary(
                audit['label'],
                audit['likelihood_prediction'],
                audit['generated_label_id'],
            ),
            'generation_seconds': audit_elapsed,
        }
    )

generation_audit = pd.concat(audit_frames, ignore_index=True)
generation_audit_summary_table = pd.DataFrame(audit_summary_rows)
zero_shot_results = zero_shot_results.merge(
    generation_audit_summary_table,
    on='test_domain',
    how='left',
    validate='one_to_one',
)

peak_gpu_memory_gb = torch.cuda.max_memory_allocated(device) / 1024 ** 3
zero_shot_results['peak_gpu_memory_gb'] = peak_gpu_memory_gb

del model, tokenizer
gc.collect()
torch.cuda.empty_cache()

generation_audit_summary_table.round(3)

## Export compact artifacts

The prediction and audit files intentionally omit the original message text. Stable hashed example IDs preserve row-level reproducibility without duplicating the datasets.

In [ ]:
prediction_export = pd.concat(
    [prediction_tables[domain] for domain in ('sms', 'enron')],
    ignore_index=True,
).drop(columns=['text'])
if 'text' in prediction_export.columns or 'text' in generation_audit.columns:
    raise RuntimeError('Raw message text must not be exported.')

artifact_frames = {
    'generative_zero_shot_results.csv': zero_shot_results,
    'generative_zero_shot_predictions.csv': prediction_export,
    'generative_zero_shot_calibration.csv': calibration_results,
    'generative_zero_shot_confidence_coverage.csv': confidence_coverage,
    'generative_generation_audit.csv': generation_audit,
}

results_directory = PROJECT_ROOT / 'results'
results_directory.mkdir(parents=True, exist_ok=True)
export_directory = Path('/kaggle/working') if IS_KAGGLE else results_directory

for filename, frame in artifact_frames.items():
    repository_path = results_directory / filename
    frame.to_csv(repository_path, index=False)
    if export_directory.resolve() != results_directory.resolve():
        frame.to_csv(export_directory / filename, index=False)
    print(f'Saved {len(frame):,} rows -> {repository_path}')
    if IS_KAGGLE:
        print(f'Download copy -> {export_directory / filename}')

## Descriptive comparison with the supervised experiments

For context, the table uses the SMS-trained supervised path: SMS to SMS for the short-message benchmark and SMS to Enron for the project transfer question. FLAN-T5 has no project training domain, so its Enron score is not described as SMS-to-Enron transfer. The seed SD of a trained model and the bootstrap interval of a fixed zero-shot model represent different uncertainty sources and their widths are not directly comparable. The table reads compatible supervised summary artifacts from `results/`.

In [ ]:
_, baseline_results = run_transfer_experiments(
    splits,
    random_state=REFERENCE_TRAINING_SEED,
)
textcnn_summary_path = PROJECT_ROOT / 'results' / 'textcnn_seed_summary.csv'
textcnn_summary = pd.read_csv(textcnn_summary_path)

comparison_rows = []
for domain in ('sms', 'enron'):
    baseline_row = baseline_results.query(
        "train_domain == 'sms' and test_domain == @domain"
    ).iloc[0]
    comparison_rows.append(
        {
            'model': 'TF-IDF + Logistic Regression',
            'test_domain': domain,
            'protocol': 'SMS-trained',
            'f1': baseline_row['f1'],
            'uncertainty': 'deterministic baseline',
        }
    )

    textcnn_row = textcnn_summary.query(
        "train_domain == 'sms' and test_domain == @domain"
    ).iloc[0]
    comparison_rows.append(
        {
            'model': 'TextCNN',
            'test_domain': domain,
            'protocol': 'SMS-trained, five seeds',
            'f1': textcnn_row['f1_mean'],
            'uncertainty': f"SD = {textcnn_row['f1_std']:.3f}",
        }
    )

    zero_row = zero_shot_results.query('test_domain == @domain').iloc[0]
    comparison_rows.append(
        {
            'model': MODEL_LABEL,
            'test_domain': domain,
            'protocol': 'zero-shot, no project training',
            'f1': zero_row['f1'],
            'uncertainty': (
                f"95% bootstrap CI "
                f"[{zero_row['f1_ci_low']:.3f}, {zero_row['f1_ci_high']:.3f}]"
            ),
        }
    )

distilbert_summary_path = PROJECT_ROOT / 'results' / 'distilbert_seed_summary.csv'
distilbert_summary = None
if distilbert_summary_path.exists():
    try:
        candidate_summary = pd.read_csv(distilbert_summary_path)
        required_columns = {
            'train_domain', 'test_domain', 'f1_mean', 'f1_std',
        }
        missing_columns = required_columns - set(candidate_summary.columns)
        if missing_columns:
            raise ValueError(
                f'missing columns: {sorted(missing_columns)}'
            )
        relevant_rows = candidate_summary.query(
            "train_domain == 'sms' and test_domain in ['sms', 'enron']"
        )
        if relevant_rows.duplicated(
            ['train_domain', 'test_domain']
        ).any():
            raise ValueError('duplicate SMS-trained domain rows')
        distilbert_summary = candidate_summary
    except (OSError, pd.errors.ParserError, ValueError) as error:
        print(
            'Skipping invalid Notebook 4 summary: '
            f'{type(error).__name__}: {error}'
        )

if distilbert_summary is not None:
    for domain in ('sms', 'enron'):
        matching = distilbert_summary.query(
            "train_domain == 'sms' and test_domain == @domain"
        )
        if len(matching) == 1:
            row = matching.iloc[0]
            comparison_rows.append(
                {
                    'model': 'DistilBERT',
                    'test_domain': domain,
                    'protocol': 'SMS-trained, five seeds',
                    'f1': row['f1_mean'],
                    'uncertainty': f"SD = {row['f1_std']:.3f}",
                }
            )
    print('Included DistilBERT summary in the descriptive comparison.')
else:
    print('Built the descriptive comparison from the selected result artifacts.')

comparison_table = pd.DataFrame(comparison_rows).sort_values(
    ['test_domain', 'f1'],
    ascending=[True, False],
    ignore_index=True,
)
comparison_table.assign(f1=comparison_table['f1'].round(3))

## Result interpretation

The statements below are generated from the executed results. They distinguish the domain gap from transfer and report uncertainty without inventing seed variability for a model that was not trained here.

In [ ]:
sms_row = zero_shot_results.query("test_domain == 'sms'").iloc[0]
enron_row = zero_shot_results.query("test_domain == 'enron'").iloc[0]
domain_gap = sms_row['f1'] - enron_row['f1']

for row in (sms_row, enron_row):
    domain = row['test_domain'].upper()
    print(
        f"{domain}: F1 {row['f1']:.3f} "
        f"(95% stratified bootstrap CI "
        f"{row['f1_ci_low']:.3f}–{row['f1_ci_high']:.3f}), "
        f"macro-F1 {row['macro_f1']:.3f}, "
        f"truncation {row['truncation_rate']:.1%}."
    )

print(f'F1 domain performance gap (SMS minus Enron): {domain_gap:+.3f}.')
for row in generation_audit_summary_table.itertuples(index=False):
    print(
        f'{row.test_domain.upper()} free-generation compliance: '
        f'{row.generation_compliance_rate:.1%}; '
        f'accuracy with invalid outputs counted wrong: '
        f'{row.generation_exact_accuracy_all_rows:.1%}.'
    )
print(
    'The confidence values are normalized over ham and spam only; '
    'calibration metrics must be interpreted with that restriction.'
)

## Scope and limitations

- FLAN-T5 was instruction-tuned externally; this project does not fine-tune or instruction-tune it.
- One fixed prompt and one verbalizer pair keep the experiment controlled, but prompt and label sensitivity are not measured.
- The length-normalized restricted-label score is sensitive to tokenization and verbalizer choice; its softmax is not a globally normalized or automatically calibrated probability.
- Public spam examples may have appeared in the model's pretraining data; that overlap cannot be verified here.
- Long messages are right-truncated after preserving the complete instruction. The measured truncation rate must accompany the results.
- Message content could imitate instructions, so prompt injection remains possible.
- The test sets were inspected in earlier work and are confirmation benchmarks, not pristine unseen holdouts.
- The free-generation audit is a balanced diagnostic subset, while the full-test restricted-label score experiment remains primary.